In [ ]:
# Importing required libraries
import cv2
import numpy as np
import face_recognition
import os
import glob
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
# Function to load images and create encodings
def load_known_faces(image_folder):
    """
    Load images from a folder and create face encodings
    Returns: known_face_encodings, known_face_names
    """
    known_face_encodings = []
    known_face_names = []
    
    # Supported image formats
    image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp']
    
    for ext in image_extensions:
        for image_path in glob.glob(os.path.join(image_folder, ext)):
            # Load image
            image = face_recognition.load_image_file(image_path)
            
            # Get face encodings
            encodings = face_recognition.face_encodings(image)
            
            if len(encodings) > 0:
                # Use the first face found
                encoding = encodings[0]
                known_face_encodings.append(encoding)
                
                # Extract name from filename (without extension)
                name = Path(image_path).stem
                known_face_names.append(name)
                print(f"Loaded: {name}")
            else:
                print(f"No face found in: {image_path}")
    
    return known_face_encodings, known_face_names

## Data Augmentation for Training Dataset
Apply various augmentations to expand your known faces dataset

In [ ]:
from scipy import ndimage
from skimage import transform
from skimage.util import random_noise

def augment_image(image, augmentation_type):
    """
    Apply various augmentation techniques to an image
    Args:
        image: numpy array of the image
        augmentation_type: type of augmentation to apply
    Returns:
        augmented image
    """
    if augmentation_type == 'flip_horizontal':
        return np.fliplr(image)
    
    elif augmentation_type == 'flip_vertical':
        return np.flipud(image)
    
    elif augmentation_type == 'rotate_90_cw':
        return np.rot90(image, k=-1)
    
    elif augmentation_type == 'rotate_90_ccw':
        return np.rot90(image, k=1)
    
    elif augmentation_type == 'rotate_180':
        return np.rot90(image, k=2)
    
    elif augmentation_type.startswith('rotate_'):
        # Random rotation between -40 and +40 degrees
        angle = np.random.uniform(-40, 40)
        return ndimage.rotate(image, angle, reshape=False, mode='nearest')
    
    elif augmentation_type == 'blur':
        # Gaussian blur with sigma up to 1.4
        sigma = np.random.uniform(0.5, 1.4)
        return ndimage.gaussian_filter(image, sigma=sigma)
    
    else:
        return image

In [ ]:
def create_augmented_dataset(input_folder, output_folder, num_augmentations=10):
    """
    Create an augmented dataset from known faces
    Args:
        input_folder: folder with original images
        output_folder: folder to save augmented images
        num_augmentations: number of augmented versions per image
    """
    os.makedirs(output_folder, exist_ok=True)
    
    # Define augmentation types
    augmentations = [
        'flip_horizontal',
        'flip_vertical', 
        'rotate_90_cw',
        'rotate_90_ccw',
        'rotate_180',
        'rotate_random',
        'blur'
    ]
    
    image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp']
    total_created = 0
    
    for ext in image_extensions:
        for image_path in glob.glob(os.path.join(input_folder, ext)):
            # Load original image
            img = cv2.imread(image_path)
            if img is None:
                continue
            
            # Resize to 640x640 (preprocessing)
            img_resized = cv2.resize(img, (640, 640))
            
            # Get filename without extension
            filename = Path(image_path).stem
            
            # Save original resized image
            original_path = os.path.join(output_folder, f"{filename}_original.jpg")
            cv2.imwrite(original_path, img_resized)
            
            # Create augmented versions
            for i in range(num_augmentations):
                # Randomly select augmentation type
                aug_type = np.random.choice(augmentations)
                
                # Apply augmentation
                augmented = augment_image(img_resized, aug_type)
                
                # Ensure image is in correct format
                augmented = np.clip(augmented, 0, 255).astype(np.uint8)
                
                # Save augmented image
                aug_filename = f"{filename}_aug_{i+1}_{aug_type}.jpg"
                aug_path = os.path.join(output_folder, aug_filename)
                cv2.imwrite(aug_path, augmented)
                
                total_created += 1
            
            print(f"Created {num_augmentations} augmentations for: {filename}")
    
    print(f"\nTotal augmented images created: {total_created}")
    print(f"Saved in: {output_folder}")

In [ ]:
# Example: Create augmented dataset
input_folder_aug = "../known_faces"  # Your known faces folder
output_folder_aug = "../augmented_faces"  # Where to save augmented images

# Uncomment to create augmented dataset
# create_augmented_dataset(input_folder_aug, output_folder_aug, num_augmentations=10)

In [ ]:
# Visualize augmentations on a sample image
def visualize_augmentations(image_path):
    """
    Display all augmentation types applied to a single image
    """
    # Load image
    img = cv2.imread(image_path)
    if img is None:
        print(f"Image not found: {image_path}")
        return
    
    # Resize to 640x640
    img = cv2.resize(img, (640, 640))
    
    augmentations = {
        'Original': img,
        'Flip Horizontal': augment_image(img, 'flip_horizontal'),
        'Flip Vertical': augment_image(img, 'flip_vertical'),
        'Rotate 90° CW': augment_image(img, 'rotate_90_cw'),
        'Rotate 90° CCW': augment_image(img, 'rotate_90_ccw'),
        'Rotate 180°': augment_image(img, 'rotate_180'),
        'Rotate Random': augment_image(img, 'rotate_random'),
        'Blur': augment_image(img, 'blur')
    }
    
    # Create subplot
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.ravel()
    
    for idx, (title, aug_img) in enumerate(augmentations.items()):
        # Convert BGR to RGB for display
        rgb_img = cv2.cvtColor(aug_img.astype(np.uint8), cv2.COLOR_BGR2RGB)
        axes[idx].imshow(rgb_img)
        axes[idx].set_title(title, fontsize=12)
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

# Test visualization with a sample image
# Replace with path to one of your images
sample_image = "../known_faces/sample.jpg"

# Uncomment to visualize
# if os.path.exists(sample_image):
#     visualize_augmentations(sample_image)

In [ ]:
# Advanced: Load known faces from augmented dataset
def load_faces_with_augmentation(base_folder, apply_augmentation=True, aug_count=5):
    """
    Load faces and optionally apply real-time augmentation
    Args:
        base_folder: folder with base images
        apply_augmentation: whether to create augmented versions
        aug_count: number of augmented versions per image
    Returns:
        known_face_encodings, known_face_names
    """
    known_face_encodings = []
    known_face_names = []
    
    augmentation_types = ['flip_horizontal', 'rotate_random', 'blur']
    image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp']
    
    for ext in image_extensions:
        for image_path in glob.glob(os.path.join(base_folder, ext)):
            # Load and resize original image
            img = cv2.imread(image_path)
            if img is None:
                continue
            
            img = cv2.resize(img, (640, 640))
            name = Path(image_path).stem
            
            # Convert to RGB for face_recognition
            rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # Get encoding for original
            encodings = face_recognition.face_encodings(rgb_img)
            if len(encodings) > 0:
                known_face_encodings.append(encodings[0])
                known_face_names.append(name)
                print(f"Loaded: {name} (original)")
            
            # Apply augmentations if enabled
            if apply_augmentation:
                for i in range(aug_count):
                    aug_type = np.random.choice(augmentation_types)
                    aug_img = augment_image(img, aug_type)
                    aug_img = np.clip(aug_img, 0, 255).astype(np.uint8)
                    
                    # Convert to RGB
                    rgb_aug = cv2.cvtColor(aug_img, cv2.COLOR_BGR2RGB)
                    
                    # Get encoding
                    aug_encodings = face_recognition.face_encodings(rgb_aug)
                    if len(aug_encodings) > 0:
                        known_face_encodings.append(aug_encodings[0])
                        known_face_names.append(name)
                
                print(f"  + Created {aug_count} augmented versions")
    
    return known_face_encodings, known_face_names

# Example: Load with augmentation
# known_face_encodings, known_face_names = load_faces_with_augmentation(
#     known_faces_path, 
#     apply_augmentation=True, 
#     aug_count=5
# )

In [ ]:
# Create a sample known faces directory (modify path as needed)
known_faces_path = "../known_faces"
os.makedirs(known_faces_path, exist_ok=True)
print(f"Place known face images in: {known_faces_path}")
print("Name each image file with the person's name (e.g., 'john_doe.jpg')")

In [ ]:
# Load known faces
print("Loading known faces...")
known_face_encodings, known_face_names = load_known_faces(known_faces_path)
print(f"\nTotal faces loaded: {len(known_face_names)}")
print(f"Names: {known_face_names}")

In [ ]:
# Function to recognize faces in an image
def recognize_faces_in_image(image_path, known_encodings, known_names, tolerance=0.6):
    """
    Recognize faces in a given image
    Args:
        image_path: path to the image
        known_encodings: list of known face encodings
        known_names: list of known face names
        tolerance: face matching tolerance (lower is more strict)
    Returns:
        image with bounding boxes and labels
    """
    # Load the image
    image = face_recognition.load_image_file(image_path)
    
    # Find all faces and their encodings in the image
    face_locations = face_recognition.face_locations(image)
    face_encodings = face_recognition.face_encodings(image, face_locations)
    
    # Convert to BGR for OpenCV
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    
    # Loop through each face found
    for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
        # Check if face matches known faces
        matches = face_recognition.compare_faces(known_encodings, face_encoding, tolerance=tolerance)
        name = "Unknown"
        confidence = 0
        
        # Calculate face distances
        face_distances = face_recognition.face_distance(known_encodings, face_encoding)
        
        if len(face_distances) > 0:
            best_match_index = np.argmin(face_distances)
            if matches[best_match_index]:
                name = known_names[best_match_index]
                confidence = 1 - face_distances[best_match_index]
        
        # Draw rectangle around the face
        color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)
        cv2.rectangle(image, (left, top), (right, bottom), color, 2)
        
        # Draw label with name and confidence
        label = f"{name} ({confidence:.2f})" if name != "Unknown" else "Unknown"
        cv2.rectangle(image, (left, bottom - 35), (right, bottom), color, cv2.FILLED)
        cv2.putText(image, label, (left + 6, bottom - 6), cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)
    
    return image, len(face_locations)

In [ ]:
# Test face recognition on a sample image
test_image_path = "test_image.jpg"  # Replace with your test image path

if os.path.exists(test_image_path):
    result_image, num_faces = recognize_faces_in_image(
        test_image_path, 
        known_face_encodings, 
        known_face_names
    )
    
    print(f"Detected {num_faces} face(s)")
    
    # Display the result
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(result_image, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title('Face Recognition Result')
    plt.show()
else:
    print(f"Test image not found: {test_image_path}")

In [ ]:
# Real-time face recognition using webcam
def recognize_faces_webcam(known_encodings, known_names):
    """
    Real-time face recognition using webcam
    Press 'q' to quit
    """
    video_capture = cv2.VideoCapture(0)
    
    if not video_capture.isOpened():
        print("Error: Could not open webcam")
        return
    
    print("Press 'q' to quit")
    
    while True:
        # Grab a single frame
        ret, frame = video_capture.read()
        
        if not ret:
            break
        
        # Resize frame for faster processing
        small_frame = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)
        rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)
        
        # Find faces
        face_locations = face_recognition.face_locations(rgb_small_frame)
        face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)
        
        # Process each face
        for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
            # Scale back up face locations
            top *= 4
            right *= 4
            bottom *= 4
            left *= 4
            
            # Check matches
            matches = face_recognition.compare_faces(known_encodings, face_encoding, tolerance=0.6)
            name = "Unknown"
            confidence = 0
            
            if len(known_encodings) > 0:
                face_distances = face_recognition.face_distance(known_encodings, face_encoding)
                best_match_index = np.argmin(face_distances)
                
                if matches[best_match_index]:
                    name = known_names[best_match_index]
                    confidence = 1 - face_distances[best_match_index]
            
            # Draw rectangle and label
            color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)
            cv2.rectangle(frame, (left, top), (right, bottom), color, 2)
            
            label = f"{name} ({confidence:.2f})" if name != "Unknown" else "Unknown"
            cv2.rectangle(frame, (left, bottom - 35), (right, bottom), color, cv2.FILLED)
            cv2.putText(frame, label, (left + 6, bottom - 6), cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)
        
        # Display
        cv2.imshow('Face Recognition', frame)
        
        # Quit on 'q'
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    video_capture.release()
    cv2.destroyAllWindows()
    print("Webcam stopped")

In [ ]:
# Start webcam face recognition
# Uncomment the line below to start real-time recognition
# recognize_faces_webcam(known_face_encodings, known_face_names)

In [ ]:
# Batch processing: Recognize faces in multiple images
def batch_recognize_faces(input_folder, output_folder, known_encodings, known_names):
    """
    Process all images in a folder and save results
    """
    os.makedirs(output_folder, exist_ok=True)
    
    image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp']
    processed = 0
    
    for ext in image_extensions:
        for image_path in glob.glob(os.path.join(input_folder, ext)):
            try:
                result_image, num_faces = recognize_faces_in_image(
                    image_path, known_encodings, known_names
                )
                
                # Save result
                filename = Path(image_path).name
                output_path = os.path.join(output_folder, f"recognized_{filename}")
                cv2.imwrite(output_path, result_image)
                
                processed += 1
                print(f"Processed: {filename} - Found {num_faces} face(s)")
            except Exception as e:
                print(f"Error processing {image_path}: {str(e)}")
    
    print(f"\nTotal images processed: {processed}")
    print(f"Results saved in: {output_folder}")

In [ ]:
# Example: Batch process images
input_folder = "test_images"  # Folder with images to test
output_folder = "recognized_images"  # Output folder

# Uncomment to run batch processing
# if os.path.exists(input_folder):
#     batch_recognize_faces(input_folder, output_folder, known_face_encodings, known_face_names)

In [ ]:
# Save face encodings for later use
import pickle

def save_encodings(encodings, names, filename='face_encodings.pkl'):
    """Save face encodings to a file"""
    data = {'encodings': encodings, 'names': names}
    with open(filename, 'wb') as f:
        pickle.dump(data, f)
    print(f"Encodings saved to {filename}")

def load_encodings(filename='face_encodings.pkl'):
    """Load face encodings from a file"""
    with open(filename, 'rb') as f:
        data = pickle.load(f)
    print(f"Loaded {len(data['names'])} encodings from {filename}")
    return data['encodings'], data['names']

In [ ]:
# Save current encodings
if len(known_face_encodings) > 0:
    save_encodings(known_face_encodings, known_face_names, 'face_encodings.pkl')
else:
    print("No encodings to save. Please load known faces first.")

In [ ]:
# Load previously saved encodings
# Uncomment to load saved encodings
# known_face_encodings, known_face_names = load_encodings('face_encodings.pkl')

## Instructions for Use:

1. **Setup Known Faces:**
   - Create a folder called `known_faces` in the parent directory
   - Add images of people you want to recognize
   - Name each file with the person's name (e.g., `john_doe.jpg`)

2. **Install Required Library:**
   ```bash
   pip install face_recognition opencv-python pillow matplotlib
   ```

3. **Run the Notebook:**
   - Execute cells in order
   - The notebook will load known faces and create encodings
   - Test on images or use webcam for real-time recognition

4. **Features:**
   - Face detection and recognition
   - Real-time webcam recognition
   - Batch image processing
   - Save/load face encodings for faster startup